# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

I'm choosing the Content Refresh Priority lane. Notebook 01 already showed a hand-written rule catching only ~24% of truly declining pages in the top 50, while a learned model caught ~74% — a real, measurable gap in a decision FlyRank's clients actually face every week: which pages to refresh first. I want to spend the next 7 weeks understanding why the model does better and turning that into something explainable, not just a black-box score.

In [14]:
# Numbers referenced below are computed live in notebooks/01_first_look_and_discovery.ipynb
# Hand-rule Precision@50: 0.240   Random forest Precision@50: 0.740   (~3.1x lift)

import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/rooman-dev/flyrank-01"
REPO_DIR = "flyrank-01"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())

Working dir: /content/flyrank-01/flyrank-01


## 2. The question: decision, action, cost of a wrong call

Decision: Which declining pages should a content team refresh this month, given limited writer/editor hours?
Unit of analysis: a page (not a client, not a query) — each row is one URL scored for refresh priority.
Action: the top-N ranked pages go into the team's refresh queue; lower-ranked pages wait.
Cost of a wrong call: a false positive wastes writer hours refreshing a page that wasn't actually declining. A false negative lets a genuinely declining, high-traffic page keep losing impressions unnoticed — the more expensive mistake, since it's a slower, quieter loss that compounds before anyone catches it.
Why ML helps: the signal isn't a single obvious threshold — Discovery A already showed search volume doesn't predict impressions, and the depth-2 tree in notebook 02 found combinations of days_since_last_update, impressions_90d, and avg_position that a flat hand rule can't capture in one line.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Quick look at the data (2-3 real numbers)

These numbers show the lane has real weight: a meaningful share of pages are declining, the obvious signal (search volume) doesn't help rank them, and there's a sizeable "stale but visible" pool worth prioritizing — exactly where a model can add value over a flat rule.

In [16]:

import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

decline_rate = df["trend_direction"].str.lower().eq("down").mean()
print(f"Share of pages currently declining: {decline_rate:.1%}")

corr = df["search_volume"].corr(df["impressions_90d"])
print(f"Correlation(search_volume, impressions_90d): {corr:.3f} — volume alone won't rank pages for us")

stale_and_visible = ((df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)).mean()
print(f"Share of pages that are stale AND still visible (candidate refresh pool): {stale_and_visible:.1%}")


Share of pages currently declining: 54.2%
Correlation(search_volume, impressions_90d): 0.001 — volume alone won't rank pages for us
Share of pages that are stale AND still visible (candidate refresh pool): 0.1%


## 4. Careful words: what I can and can't claim

I can say a page is observed to be declining in impressions over the measured window, and that certain features are associated with decline in this anonymized sample. I cannot claim to know why Google's ranking algorithm changed, cannot promise a refresh will reverse the trend, and cannot generalize beyond FlyRank's client mix without re-validating on new data. Every recommendation is decision-support, not a guarantee.

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
   - [x] The notebook runs top to bottom with no errors (Runtime → Run all)
   - [x] No client names, URLs, or private queries anywhere
   - [x] My claims use careful words: observed, measured, directional, decision-support
   - [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.